# Notebook 6: The `ltu-ili` Pipeline

[`ltu-ili`](https://github.com/maho3/ltu-ili) (the "Learning the Universe Implicit Likelihood Inference" pipeline, [Ho et al., 2024](https://arxiv.org/abs/2402.05137)) is widely used in cosmology. It does not implement new inference methods. It is a common interface on top of existing packages (`sbi`, `lampe` and `pydelfi`), designed for reproducible analyses:

- one interface for several backends,
- **ensembles** of networks as the default,
- a standard set of **validation** plots,
- complete analyses defined in **configuration files**.

We solve the Gaussian toy problem of Notebooks 4 and 5 once more. This notebook is both a continuation of the tutorial and a quickstart for the package.

**What you will learn**
- the `ltu-ili` workflow: data loader, prior, networks, `InferenceRunner`
- how to work with an ensemble posterior
- which validation plots `ltu-ili` produces and how to read them
- how to run NLE and how to switch the backend
- how to define an analysis with configuration files

**Runtime:** about three minutes on a laptop.

*Version note:* this notebook was written for `ltu-ili` 0.1.5 with `sbi` 0.27. `ltu-ili` uses the old class names of `sbi` (`SNPE`, `SNLE`), which will be removed in `sbi` 0.28, so do not upgrade `sbi` beyond 0.27 without checking for a new release of `ltu-ili`. The documentation consists of the [API reference](https://ltu-ili.readthedocs.io/) and of the notebooks and `CONFIG.md` in the repository. The import name of the package is `ili`.

In [ ]:
import os
import time
import logging
import tempfile
import numpy as np
import torch
import matplotlib.pyplot as plt

import ili
from ili.dataloaders import NumpyLoader, StaticNumpyLoader
from ili.inference import InferenceRunner
from ili.validation.metrics import PlotSinglePosterior, PosteriorCoverage
from ili.validation.runner import ValidationRunner

# The toy problem from Notebook 4
from tutorial_utils import gaussian_benchmark, analytic_moments, plot_posterior

# ili switches on INFO messages for all packages when it is imported. We switch them off again.
logging.getLogger().setLevel(logging.WARNING)

torch.manual_seed(42)
np.random.seed(42)

# Our models are small, so they do not benefit from many CPU threads. Limiting the thread
# count avoids large slow-downs from thread oversubscription (common on laptops).
torch.set_num_threads(4)

## 1. Data Loader and Prior

`ltu-ili` separates an analysis into three stages: **data**, **inference** and **validation**.

**Data.** The simulations are passed through a *data loader*. For arrays in memory this is `NumpyLoader`. There are also loaders for files on disk (`StaticNumpyLoader`) and for simulators which are run on the fly (`SBISimulator`, needed for sequential methods).

**Prior.** Use the distributions in `ili.utils` (`Uniform`, `IndependentNormal`, `IndependentTruncatedNormal`, `MultivariateNormal`, ...). They are PyTorch distributions which treat all parameters as one event, which is what the backends expect. `Uniform` needs one entry per parameter.

In [ ]:
benchmark = gaussian_benchmark()
theta_train, x_train = benchmark.theta_train.numpy(), benchmark.x_train.numpy()
theta_test, x_test = benchmark.theta_test.numpy(), benchmark.x_test.numpy()
theta_true, x_obs = benchmark.theta_obs, benchmark.x_obs

# Data loader for the training simulations: x [N, num_data], theta [N, num_parameters]
loader = NumpyLoader(x=x_train, theta=theta_train)

# Uniform prior over mean [-5, 5] and log_std [-2, 1]
prior = ili.utils.Uniform(low=[-5.0, -2.0], high=[5.0, 1.0], device='cpu')

labels = ['Mean', 'Log Std']

## 2. Training an Ensemble with NPE

The inference stage is configured with `InferenceRunner.load`:
- `backend`: `'sbi'`, `'lampe'` or `'pydelfi'`
- `engine`: `'NPE'`, `'NLE'` or `'NRE'` (and the sequential versions `'SNPE'`, `'SNLE'`, `'SNRE'`)
- `nets`: a list of networks. **All of them are trained and combined into an ensemble.**
- `train_args`: batch size, learning rate, early stopping

The networks are created with `ili.utils.load_nde_sbi` (for the `sbi` backend). The available models for NPE and NLE are `'maf'`, `'nsf'`, `'mdn'` and `'made'`, with the arguments `hidden_features` and `num_transforms` (`num_components` for the `'mdn'`).

Why an ensemble? Different architectures (and different random initializations) make different errors. Averaging over several networks makes the result more robust, and comparing the members with each other is a cheap sanity check. Here we combine a MAF and a spline flow.

In [ ]:
# Each call of load_nde_sbi returns a list of networks (use repeats=n to get n copies)
nets = [
    ili.utils.load_nde_sbi(engine='NPE', model='maf', hidden_features=50, num_transforms=5),
    ili.utils.load_nde_sbi(engine='NPE', model='nsf', hidden_features=64, num_transforms=4),
]

train_args = {
    'training_batch_size': 200,
    'learning_rate': 5e-4,
    'stop_after_epochs': 20,     # early stopping
}

runner = InferenceRunner.load(
    backend='sbi',
    engine='NPE',
    prior=prior,
    nets=nets,
    device='cpu',
    train_args=train_args,
    out_dir=None,                # a directory here would save the posterior and a training summary to disk
)

start = time.time()
posterior_ensemble, summaries = runner(loader=loader, seed=1)
print(f'\nTraining time: {time.time() - start:.0f} s')

The runner returns the posterior and one summary per network with the training history. `ltu-ili` stores log-probabilities, i.e. the negative loss, so higher is better.

*A detail of the versions used here:* the `sbi` backend first trains every network once with the default settings of `sbi` and then continues with your `train_args`. This is why the summaries contain two entries for `epochs_trained`, and why settings like `max_num_epochs` do not limit the total training time.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, summary in zip(['MAF', 'NSF'], summaries):
    line, = ax.plot(summary['training_log_probs'], ls=':')
    ax.plot(summary['validation_log_probs'], color=line.get_color(), label=name)
    print(f"{name}: epochs trained {summary['epochs_trained']}, best validation log-probability {summary['best_validation_log_prob'][-1]:.3f}")
ax.set(xlabel='Epoch', ylabel=r'$\log q(\theta|x)$', ylim=(-1.5, 0.6), title='Validation (solid) and training (dotted)')
ax.legend()
plt.show()

## 3. The Ensemble Posterior

The posterior is an `EnsemblePosterior` from `sbi`. It behaves like the posterior objects of Notebook 5:
- `sample((n,), x)` draws every sample from one of the members, chosen at random according to the weights,
- `log_prob(theta, x)` is the weighted mixture of the members,
- `weights` are the weights of the members. They are computed from the validation log-probabilities: better networks get more weight.
- `posteriors` is the list of the individual members, which can be used on their own.

In [ ]:
print('Ensemble weights (MAF, NSF):', posterior_ensemble.weights.numpy().round(3))

samples_ensemble = posterior_ensemble.sample((5000,), x_obs, show_progress_bars=False)
samples_maf = posterior_ensemble.posteriors[0].sample((5000,), x_obs, show_progress_bars=False)
samples_nsf = posterior_ensemble.posteriors[1].sample((5000,), x_obs, show_progress_bars=False)

exact_mean, exact_std = analytic_moments(x_obs)
print('\n                 posterior mean       posterior std')
for name, s in (('Ensemble', samples_ensemble), ('MAF', samples_maf), ('NSF', samples_nsf)):
    print(f'{name:10s}      ', s.mean(dim=0).numpy().round(3), '   ', s.std(dim=0).numpy().round(3))
print(f"{'Exact':10s}      ", exact_mean.numpy().round(3), '   ', exact_std.numpy().round(3))

print('\nlog q(theta_true | x_obs) =', posterior_ensemble.log_prob(theta_true.unsqueeze(0), x_obs).item())

plot_posterior({'Ensemble': samples_ensemble, 'MAF': samples_maf, 'NSF': samples_nsf}, x_obs, theta_true)

Always look at the members individually as well. If one network has failed to train properly, the ensemble inherits its error (with a small weight), and a look at the individual posteriors reveals this immediately.

## 4. Validation

The validation stage consists of *metrics* in `ili.validation.metrics`. A metric is created with its settings and then called with the posterior and test data. The settings which all metrics share:
- `num_samples`: number of posterior samples per test simulation
- `sample_method`: `'direct'` for NPE. For NLE the options are `'emcee'`, `'vi'` or the name of an MCMC method of `sbi` (e.g. `'slice_np_vectorized'`), configured with `sample_params`. **The default is `'emcee'`**, which also works for NPE but is much slower, so always set `'direct'` for NPE.
- `labels`: names of the parameters
- `out_dir`: where to save the figures. With `None` the figures are only returned.

### A Single Posterior

`PlotSinglePosterior` draws a corner plot for one observation.

In [ ]:
metric = PlotSinglePosterior(num_samples=2000, sample_method='direct', labels=labels, out_dir=None)
grid = metric(posterior=posterior_ensemble, x_obs=x_obs, theta_fid=theta_true)

### Coverage

`PosteriorCoverage` samples the posteriors of all test simulations and produces the plots listed in `plot_list`:

- `'coverage'`: for every parameter separately, how often the true value falls into the central credible interval of a given level. Should follow the diagonal.
- `'histogram'`: the rank of the true value among the posterior samples, for every parameter. This is the SBC rank histogram from Notebook 5 and should be flat (peaked in the middle: posteriors too wide, U-shaped: too narrow).
- `'predictions'`: posterior mean and standard deviation against the true value for every test simulation. Should scatter around the diagonal. This plot shows directly how well each parameter is constrained.
- `'tarp'`: the TARP coverage test of all parameters together, with an uncertainty band from bootstrapping.

(There is a fifth plot, `'logprob'`, which does not work with NumPy 2.)

In [ ]:
metric = PosteriorCoverage(
    num_samples=1000,
    sample_method='direct',
    labels=labels,
    plot_list=['coverage', 'histogram', 'predictions', 'tarp'],
    out_dir=None,
)
start = time.time()
figures = metric(posterior=posterior_ensemble, x=x_test, theta=theta_test)
print(f'Coverage of {len(x_test)} test simulations: {time.time() - start:.1f} s')

## 5. NLE with `ltu-ili`

Switching to NLE only requires a different `engine`. The posterior then runs an MCMC when we ask for samples. Keyword arguments of `sample` are passed on to the MCMC sampler of `sbi` (see Notebook 5).

In [ ]:
runner_nle = InferenceRunner.load(
    backend='sbi',
    engine='NLE',
    prior=prior,
    nets=[ili.utils.load_nde_sbi(engine='NLE', model='maf', hidden_features=50, num_transforms=5)],
    device='cpu',
    train_args=train_args,
)
posterior_nle, summaries_nle = runner_nle(loader=loader, seed=1)

start = time.time()
samples_nle = posterior_nle.sample((5000,), x_obs, method='slice_np_vectorized', num_chains=50, warmup_steps=100,
                                   show_progress_bars=False)
print(f'\nNLE: {time.time() - start:.1f} s for 5000 posterior samples')

plot_posterior({'NPE ensemble': samples_ensemble, 'NLE': samples_nle}, x_obs, theta_true)

The validation metrics work for NLE as well, but they need an MCMC run (or a variational fit) for every test simulation. As a demonstration, we run the coverage test for only 20 test simulations, with `emcee` as sampler. For a meaningful test we would need a few hundred, which takes correspondingly longer. We come back to this problem in Notebooks 7 and 8.

In [ ]:
metric_nle = PosteriorCoverage(
    num_samples=200,
    sample_method='emcee',
    sample_params={'num_chains': 8, 'burn_in': 100, 'thin': 1},
    labels=labels,
    plot_list=['predictions'],
    out_dir=None,
)
start = time.time()
figures_nle = metric_nle(posterior=posterior_nle, x=x_test[:20], theta=theta_test[:20])
print(f'NLE, 20 test simulations: {time.time() - start:.1f} s')

## 6. Switching the Backend

The same analysis can be run with the [`lampe`](https://lampe.readthedocs.io/) backend, which uses the flows of the `zuko` library (NPE only). The networks are then created with `ili.utils.load_nde_lampe`, everything else stays the same. (The third backend, `pydelfi`, needs TensorFlow 1 and is not installed here.)

In [ ]:
runner_lampe = InferenceRunner.load(
    backend='lampe',
    engine='NPE',
    prior=prior,
    nets=[ili.utils.load_nde_lampe(model='nsf', hidden_features=64, num_transforms=4)],
    device='cpu',
    train_args=train_args,
)
start = time.time()
posterior_lampe, summaries_lampe = runner_lampe(loader=loader, seed=1)
print(f'\nTraining time: {time.time() - start:.0f} s')

samples_lampe = posterior_lampe.sample((5000,), x_obs, show_progress_bars=False)
plot_posterior({'NPE, sbi backend (ensemble)': samples_ensemble, 'NPE, lampe backend (NSF)': samples_lampe}, x_obs, theta_true)

## 7. Configuration Files

For real analyses, `ltu-ili` is usually driven by YAML configuration files: one for the data, one for the inference and one for the validation. The configuration is then a complete, readable record of the analysis which can be put under version control and run on a cluster with a three-line script.

Below we write the three files for our problem into a temporary directory and run them. To keep it fast, we train a single spline flow on half of the simulations.

In [ ]:
work_dir = tempfile.mkdtemp()

# The data stage reads .npy files from disk
np.save(f'{work_dir}/x_train.npy', x_train[:5000])
np.save(f'{work_dir}/theta_train.npy', theta_train[:5000])
np.save(f'{work_dir}/x_test.npy', x_test[:200])
np.save(f'{work_dir}/theta_test.npy', theta_test[:200])

data_config = f"""
in_dir: '{work_dir}'
x_file: 'x_train.npy'
theta_file: 'theta_train.npy'
"""

inference_config = f"""
prior:
  module: 'ili.utils'
  class: 'Uniform'
  args:
    low: [-5.0, -2.0]
    high: [5.0, 1.0]

model:
  backend: 'sbi'
  engine: 'NPE'
  name: 'gaussian_toy'
  nets:
    - model: 'nsf'
      hidden_features: 64
      num_transforms: 4
      signature: 'nsf'

train_args:
  training_batch_size: 200
  learning_rate: 0.0005
  stop_after_epochs: 20

device: 'cpu'
out_dir: '{work_dir}'
"""

validation_config = f"""
out_dir: '{work_dir}'
posterior_file: 'gaussian_toy_posterior.pkl'
labels: ['Mean', 'Log Std']
ensemble_mode: True

metrics:
  coverage:
    module: 'ili.validation.metrics'
    class: 'PosteriorCoverage'
    args:
      num_samples: 500
      sample_method: 'direct'
      plot_list: ['coverage', 'tarp']
"""

for name, config in (('data', data_config), ('inference', inference_config), ('validation', validation_config)):
    with open(f'{work_dir}/{name}.yaml', 'w') as handle:
        handle.write(config)

print(inference_config)

Running the analysis from the configuration files:

In [ ]:
# Inference: trains the networks and saves the posterior and a summary in out_dir
config_loader = StaticNumpyLoader.from_config(f'{work_dir}/data.yaml')
config_runner = InferenceRunner.from_config(f'{work_dir}/inference.yaml')
config_runner(loader=config_loader)

# Validation: loads the saved posterior, runs the metrics on the test data and saves the plots in out_dir
test_loader = StaticNumpyLoader(in_dir=work_dir, x_file='x_test.npy', theta_file='theta_test.npy')
validation_runner = ValidationRunner.from_config(f'{work_dir}/validation.yaml')
validation_runner(loader=test_loader)
plt.close('all')     # the plots are saved to disk, we do not need to show them again

print('\nFiles in the output directory:')
for file_name in sorted(os.listdir(work_dir)):
    print('  ', file_name)

The saved posterior is a pickle file and can be loaded in any later session:

In [ ]:
import pickle

with open(f'{work_dir}/gaussian_toy_posterior.pkl', 'rb') as handle:
    posterior_loaded = pickle.load(handle)

samples_loaded = posterior_loaded.sample((5000,), x_obs, show_progress_bars=False)
print('Posterior mean:', samples_loaded.mean(dim=0).numpy().round(3), ' exact:', exact_mean.numpy().round(3))
print('Posterior std: ', samples_loaded.std(dim=0).numpy().round(3), ' exact:', exact_std.numpy().round(3))

## Cheat Sheet

| Step | Code |
|---|---|
| Data | `loader = NumpyLoader(x=x, theta=theta)` |
| Prior | `prior = ili.utils.Uniform(low=[...], high=[...], device='cpu')` |
| Networks | `nets = [ili.utils.load_nde_sbi(engine='NPE', model='maf', hidden_features=50, num_transforms=5), ...]` |
| Inference | `runner = InferenceRunner.load(backend='sbi', engine='NPE', prior=prior, nets=nets, train_args={...})`<br>`posterior, summaries = runner(loader=loader)` |
| Samples | `posterior.sample((n,), x_obs)`, members: `posterior.posteriors[i]`, weights: `posterior.weights` |
| Validation | `PosteriorCoverage(num_samples=1000, sample_method='direct', labels=labels, plot_list=[...])(posterior=posterior, x=x_test, theta=theta_test)` |
| Configuration files | `InferenceRunner.from_config('inference.yaml')`, `ValidationRunner.from_config('validation.yaml')` |

Things to keep in mind:
- Use the priors from `ili.utils`, not plain `torch.distributions`.
- Set `sample_method='direct'` in the validation metrics for NPE.
- An embedding network (e.g. `ili.embedding.FCN`, or any PyTorch module) can be passed to `load_nde_sbi` with `embedding_net=...` to compress large data vectors.
- Like `sbi`, the `sbi` backend writes TensorBoard logs to a folder `sbi-logs` in the current directory.

## Summary

- `ltu-ili` wraps `sbi` (and other backends) in a pipeline with three stages: data, inference, validation.
- Ensembles of several networks are the default. Check the members individually.
- The validation metrics provide the standard set of calibration plots with a few lines of code.
- A complete analysis can be written down in configuration files, which makes it reproducible.
- Validation of NLE posteriors is slow, because every test simulation needs its own MCMC run.

**Next:** Notebook 7 turns to a harder problem, a large data vector and training simulations which do not follow the prior, and goes back to writing things by hand to see what is needed to solve it.